# SRM Melhoria de Processo

## Bibliotecas Usadas

In [24]:
import pandas as pd
import time
import numpy as np

In [25]:
 # Parametros

periodo = 3
ano = 2026

## Montagem da Base N13P

In [26]:
#-#-#-# Inicio da contagem do tempo de execução #-#-#-#
inicio_total = time.time()
inicio_ciclo = time.time()


# Ciclo

colunas_base = ['Regional', 'GP', 'Gerente', 'Rede', 'COD_CLIENTE', 'Company Code','CD',
 'NOME_CLIENTE', 'UF', 'Região', 'EAN', 'SKU', 'Desc. SKU', 'Classificação', 'Marca']

colunas_periodos = [f'P{i:02d}-{ano}' for i in range(periodo, 14)]

colunas = colunas_base + colunas_periodos

tipos_colunas ={'EAN': str, 
                'COD_CLIENTE': str,
                'Company Code': str, 
                'CD': str, 
                'SKU': str}

df_ciclo_n13 = pd.read_excel(f'../data/Arquivos/Ciclo_P{periodo:02d} N13P {ano} - envio.xlsx', header=3, usecols=colunas,
                             dtype=tipos_colunas,
                             engine='calamine')


# Clientes

tipos_colunas_clientes = {'COD_CLIENTE': str, 'NOME_CLIENTE': str, 'COD REDE': str, 'COD SUBREDE': str, 'COND. PAG': str}

df_clientes = pd.read_excel('../data/Arquivos/BASE CLIENTES.xlsx', 
                            dtype=tipos_colunas_clientes,
                            engine='calamine')


# Produtos

tipos_colunas ={'EAN': str, 
                'SKU': str,
                'Ton/CDA': float, 
                'Unid/\nCX	': float, 
                'Hierarquia': str,
                'NCM': str,
                'kg/Un': float,
                'H05': str,
                'LSV': float}

df_produtos = pd.read_excel('../data/Arquivos/BASE PRODUTOS.xlsx', 
                            dtype=tipos_colunas,
                            engine='calamine')

df_produtos = df_produtos.rename(columns={'Unid/\nCX': 'Unid/CDA', 'kg/Un': 'kg/UN'})

#-#-#-# Colunas especificas #-#-#-#

#  UF Origem

dicionario_uf = {
    'BR01': 'SP',
    'BR03': 'PE',
    'BR30': 'SP',
    'BR31': 'MG',
}

df_ciclo_n13['UF ORIGEM'] = df_ciclo_n13['CD'].map(dicionario_uf)

df_ciclo_n13[['CD', 'UF ORIGEM']].head(10)

# COD GP

dicionario_gp = {
    'ATACADO CASH & CARRY': 'AG',
    'GPA': 'AH',
    "SAM'S CLUB": 'AM',
    'GROCERY': 'AL',
    'ASSAI': 'AX',
    'Atacadão': 'TA',
    'DIST. MISTO': 'BI',
    'ESPECIALISTA DIRETO': 'AD',
    'DIA %': 'AF',
    'DIST. ALIMENTAR': 'AI',
    'DIST. ESPECIALISTA': 'AJ',
    'CENCOSUD': 'BJ',
    'CARREFOUR': 'AE',
    'ECOMMERCE': 'BO',
    'KA ESPECIALISTA': 'AQ',
    'PETZ': 'BL',
    'ATACADOS': 'AB',
    'MARTINS': 'BK',
    'COBASI': 'BM',
    'ATACADOS ESPECIAIS': 'BT',
    'CONVENIENCIAS': 'BP'
}

df_ciclo_n13['CÓD GP'] = df_ciclo_n13['GP'].map(dicionario_gp)

df_ciclo_n13['GP'] = df_ciclo_n13['GP'].str.strip()

# EAN Espelho

search_ean = df_produtos.set_index('EAN', drop=False)['EAN'].to_dict()
df_ciclo_n13['EAN Espelho'] = df_ciclo_n13['EAN'].map(search_ean)

search_desc = df_produtos.set_index('Descrição', drop=False)['EAN'].to_dict()
secondary_search = df_ciclo_n13['Desc. SKU'].map(search_desc)
df_ciclo_n13['EAN Espelho'] = df_ciclo_n13['EAN Espelho'].fillna(secondary_search)

# SKU

colunas_produtos = ['EAN', 'SKU', 'Family Price', 'Hierarquia', 'Class.', 'NCM', 
                    'Origem', 'kg/UN', 'Ton/CDA', 'Unid/CDA', 'LSV']

df_prod_exato = df_produtos[colunas_produtos].drop_duplicates(subset=['EAN', 'SKU'], keep='first')
df_prod_resgate = df_produtos[colunas_produtos].drop(columns=['SKU']).drop_duplicates(subset=['EAN'], keep='first')

df_ciclo_n13 = pd.merge(df_ciclo_n13,
                        df_prod_exato,
                        left_on=['EAN Espelho', 'SKU'],
                        right_on=['EAN', 'SKU'],
                        how='left')

df_ciclo_n13 = df_ciclo_n13.drop(columns=['EAN_y'], errors='ignore')
df_ciclo_n13 = df_ciclo_n13.rename(columns={'EAN_x': 'EAN'})

df_ciclo_n13 = pd.merge(df_ciclo_n13,
                        df_prod_resgate,
                        left_on='EAN Espelho',
                        right_on='EAN',
                        how='left',
                        suffixes=('', '_resgate'))

colunas_preencher = ['Family Price', 'Hierarquia', 'Class.', 'NCM', 'Origem', 'kg/UN', 'Ton/CDA', 'Unid/CDA', 'LSV']

for col in colunas_preencher:
    df_ciclo_n13[col] = df_ciclo_n13[col].fillna(df_ciclo_n13[f'{col}_resgate'])

colunas_lixo = [f'{col}_resgate' for col in colunas_preencher] + ['EAN_resgate']
df_ciclo_n13 = df_ciclo_n13.drop(columns=colunas_lixo, errors='ignore')

# Codigo Subrede

df_clientes_limpo = df_clientes[['COD_CLIENTE', 'COD REDE','COD SUBREDE', 'COND. PAG']].drop_duplicates(subset=['COD_CLIENTE'], keep='first')

df_ciclo_n13 = pd.merge(
    df_ciclo_n13,
    df_clientes_limpo, 
    on='COD_CLIENTE',
    how='left'
)

fim_ciclo = time.time()
tempo_ciclo = fim_ciclo - inicio_ciclo

In [27]:
# for col in df_ciclo_n13.columns:
#     print(f"{col}: {df_ciclo_n13[col].dtype}")

## Calculos

### ZP55

In [28]:
inicio_zp55 = time.time()

# Importação

tipos_colunas = {'CHAVE' : str, 'Cadastro': float}

colunas = ['CHAVE', 'Cadastro']

df_zp55 = pd.read_excel('../data/Arquivos/ZP55.xlsx', 
                        header=1,
                        usecols=colunas,
                        dtype=tipos_colunas,
                        engine='calamine')

df_zp55['Cadastro'] = df_zp55['Cadastro'].round(4)

# Busca

df_zp55['CHAVE'] = df_zp55['CHAVE'].astype(str).str.strip()
dic_zp55 = df_zp55.drop_duplicates(subset=['CHAVE'], keep='first').set_index('CHAVE')['Cadastro'].to_dict()


chave_1_full = df_ciclo_n13['Company Code'].astype(str).str.strip() + '_' + df_ciclo_n13['COD_CLIENTE'].astype(str).str.strip() + '_' + df_ciclo_n13['Hierarquia'].astype(str).str.strip()
chave_1_10 = df_ciclo_n13['Company Code'].astype(str).str.strip() + '_' + df_ciclo_n13['COD_CLIENTE'].astype(str).str.strip() + '_' + df_ciclo_n13['Hierarquia'].astype(str).str.strip().str[:10]

chave_2 = df_ciclo_n13['CD'].astype(str).str.strip() + '_' + df_ciclo_n13['UF'].astype(str).str.strip() + '_' + df_ciclo_n13['Origem'].astype(str).str.strip()
chave_3 = df_ciclo_n13['CD'].astype(str).str.strip() + '_' + df_ciclo_n13['UF'].astype(str).str.strip() + '_' + df_ciclo_n13['NCM'].astype(str).str.strip()
chave_4 = df_ciclo_n13['CD'].astype(str).str.strip() + '_' + df_ciclo_n13['UF'].astype(str).str.strip() + '_' + df_ciclo_n13['Hierarquia'].astype(str).str.strip().str[:10]

df_ciclo_n13['CLIENTE'] = (chave_1_full.map(dic_zp55).fillna(chave_1_10.map(dic_zp55)) / 100)
df_ciclo_n13['CD + UF DESTINO + Importação'] = (chave_2.map(dic_zp55) / 100)
df_ciclo_n13['CD + UF DESTINO + NCM'] = (chave_3.map(dic_zp55) / 100)
df_ciclo_n13['CD + UF DESTINO + H05'] = (chave_4.map(dic_zp55) / 100)

df_ciclo_n13['ZP55'] = (df_ciclo_n13['CLIENTE']
                        .fillna(df_ciclo_n13['CD + UF DESTINO + Importação'])
                        .fillna(df_ciclo_n13['CD + UF DESTINO + NCM'])
                        .fillna(df_ciclo_n13['CD + UF DESTINO + H05'])
                        )

df_ciclo_n13['ZP55'] = df_ciclo_n13['ZP55'].round(4)

fim_zp55 = time.time()
tempo_zp55 = fim_zp55 - inicio_zp55

### ZP 54

In [29]:
inicio_zp54 = time.time()

# Importação

tipos_colunas = {'CHAVE' : str, 'Cadastro': float}

colunas = ['CHAVE', 'Cadastro']

df_zp54 = pd.read_excel('../data/Arquivos/ZP54.xlsx', 
                        header=1,
                        usecols=colunas,
                        dtype=tipos_colunas,
                        engine='calamine')

df_zp54['Cadastro'] = df_zp54['Cadastro'].round(4)

# Chaves

dic_zp54 = df_zp54.set_index('CHAVE')['Cadastro'].to_dict()

chave_1_12 = df_ciclo_n13['Company Code'].astype(str) + '_' + df_ciclo_n13['COD_CLIENTE'].astype(str) + '_' + df_ciclo_n13['Hierarquia'].astype(str).str[:12]
chave_2 = df_ciclo_n13['Company Code'].astype(str) + '_' + df_ciclo_n13['COD SUBREDE'].astype(str) + '_' + df_ciclo_n13['Hierarquia'].astype(str).str[:12]
chave_3 = df_ciclo_n13['Company Code'].astype(str) + '_' + df_ciclo_n13['CÓD GP'].astype(str) + ' ' + df_ciclo_n13['UF'].astype(str) + '_' + df_ciclo_n13['Hierarquia'].astype(str).str[:12]
chave_4 = df_ciclo_n13['Company Code'].astype(str) + '_' +df_ciclo_n13['CÓD GP'].astype(str) + ' ' + df_ciclo_n13['UF'].astype(str) + '_' + df_ciclo_n13['Hierarquia'].astype(str).str[:10]

# Busca

df_ciclo_n13['1. CLIENTE'] = (chave_1_12.map(dic_zp54) / 100).round(4)
df_ciclo_n13['1. REDE'] = (chave_2.map(dic_zp54) / 100).round(4)
df_ciclo_n13['1. GP UF HIER 6'] = (chave_3.map(dic_zp54) / 100).round(4)
df_ciclo_n13['1. GP UF HIER 5'] = (chave_4.map(dic_zp54) / 100).round(4)

df_ciclo_n13['ZP54'] = (df_ciclo_n13['1. CLIENTE']
                        .fillna(df_ciclo_n13['1. REDE'])
                        .fillna(df_ciclo_n13['1. GP UF HIER 6'])
                        .fillna(df_ciclo_n13['1. GP UF HIER 5'])
                        )

df_ciclo_n13['ZP54'] = df_ciclo_n13['ZP54'].round(4)

fim_zp54 = time.time()
tempo_zp54 = fim_zp54 - inicio_zp54

### GSVs

In [30]:
inicio_gsv = time.time()

# GSV/CDA

df_ciclo_n13['GSV/CDA'] = df_ciclo_n13['LSV'] * (1 + df_ciclo_n13['ZP55']) * (1 + df_ciclo_n13['ZP54'])

df_ciclo_n13['GSV/CDA'] = df_ciclo_n13['GSV/CDA'].round(4)

# GSV/TON

coluna_BA = df_ciclo_n13['GSV/CDA'] 
coluna_AA = df_ciclo_n13['kg/UN'] 
coluna_AO = df_ciclo_n13['Unid/CDA'] 

denominador = coluna_AA * coluna_AO

df_ciclo_n13['GSV/TON'] = np.where(
    (denominador == 0) | (denominador.isna()),
    np.nan,                                   
    (coluna_BA / denominador) * 1000           
)

df_ciclo_n13['GSV/TON'] = df_ciclo_n13['GSV/TON'].round(4)


fim_gsv = time.time()
tempo_gsv = fim_gsv - inicio_gsv

### Projeções

In [31]:
inicio_projecao = time.time()

for p in colunas_periodos:
    nome_coluna_projecao = f'GSV R$ {p} - {ano}'

    df_ciclo_n13[nome_coluna_projecao] = df_ciclo_n13[p] * df_ciclo_n13['GSV/TON']

fim_projecao = time.time()
tempo_projecao = fim_projecao - inicio_projecao

### ZP53

In [32]:
inicio_zp53 = time.time()

# Importação

tipos_colunas = {'CHAVE' : str, 'Cadastro': float, 'P\'ANO_FIM' : str}

colunas = ['CHAVE', 'Cadastro', 'P\'ANO_FIM']

df_zp53 = pd.read_excel('../data/Arquivos/ZP53.xlsx', 
                        header=1,
                        usecols=colunas,
                        dtype=tipos_colunas,
                        engine='calamine')

df_zp53['Cadastro'] = df_zp53['Cadastro'].round(2)

df_zp53['CHAVE'] = df_zp53['CHAVE'].astype(str).str.strip()

# Chaves

dic_zp531 = df_zp53.drop_duplicates(subset=['CHAVE'], keep='first').set_index('CHAVE')['Cadastro'].to_dict()
dic_zp532 = df_zp53.drop_duplicates(subset=['CHAVE'], keep='first').set_index('CHAVE')['P\'ANO_FIM'].to_dict()

chave_1 = df_ciclo_n13['Company Code'].astype(str).str.strip() + '_' + df_ciclo_n13['COD_CLIENTE'].astype(str).str.strip() + '_' + df_ciclo_n13['Hierarquia'].astype(str).str.strip()
chave_2 = df_ciclo_n13['Company Code'].astype(str).str.strip() + '_' + df_ciclo_n13['COD SUBREDE'].astype(str).str.strip() + '_' + df_ciclo_n13['Hierarquia'].astype(str).str.strip()
chave_3 = df_ciclo_n13['Company Code'].astype(str).str.strip() + '_' + df_ciclo_n13['CÓD GP'].astype(str).str.strip() + ' ' + df_ciclo_n13['UF'].astype(str).str.strip() + '_' + df_ciclo_n13['Hierarquia'].astype(str).str.strip()
chave_4 = df_ciclo_n13['Company Code'].astype(str).str.strip() + '_' + df_ciclo_n13['CÓD GP'].astype(str).str.strip() + '_' + df_ciclo_n13['Hierarquia'].astype(str).str.strip().str[:10] # <-- CORREÇÃO AQUI!

# Busca
df_ciclo_n13['1. EMISSOR'] = chave_1.map(dic_zp531) / 100
df_ciclo_n13['1. REDE'] = chave_2.map(dic_zp531) / 100
df_ciclo_n13['1. GP UF'] = chave_3.map(dic_zp531) / 100
df_ciclo_n13['1. GP'] = chave_4.map(dic_zp531) / 100

df_ciclo_n13['2. EMISSOR'] = chave_1.map(dic_zp532)
df_ciclo_n13['2. REDE'] = chave_2.map(dic_zp532)
df_ciclo_n13['2. GP UF'] = chave_3.map(dic_zp532)
df_ciclo_n13['2. GP'] = chave_4.map(dic_zp532)

df_ciclo_n13['ZP53'] = (df_ciclo_n13['1. EMISSOR']
                        .fillna(df_ciclo_n13['1. REDE'])
                        .fillna(df_ciclo_n13['1. GP UF'])
                        .fillna(df_ciclo_n13['1. GP'])
                        .fillna(0) 
                    )

df_ciclo_n13['ZP53'] = df_ciclo_n13['ZP53'].round(4)

fim_zp53 = time.time()
tempo_zp53 = fim_zp53 - inicio_zp53

### ZP52

In [33]:
inicio_zp52 = time.time()

# Importação

tipos_colunas = {'CHAVE' : str, 'Cadastro': float}

colunas = ['CHAVE', 'Cadastro']

df_zp52 = pd.read_excel('../data/Arquivos/ZP52.xlsx', 
                        header=1,
                        usecols=colunas,
                        dtype=tipos_colunas,
                        engine='calamine')

df_zp52['Cadastro'] = df_zp52['Cadastro'].round(2)

df_zp52['CHAVE'] = df_zp52['CHAVE'].astype(str).str.strip()

# Chaves

dic_zp52 = df_zp52.set_index('CHAVE')['Cadastro'].to_dict()

chave_1 = df_ciclo_n13['Company Code'].astype(str) + '_' + df_ciclo_n13['COD_CLIENTE'].astype(str) + '_' + df_ciclo_n13['Hierarquia'].astype(str).str[:8]
chave_2 = df_ciclo_n13['Company Code'].astype(str) + '_' + df_ciclo_n13['COD SUBREDE'].astype(str) + '_' + df_ciclo_n13['Hierarquia'].astype(str).str[:2]

# Busca
df_ciclo_n13['H04'] = (chave_1.map(dic_zp52) / 100).round(4)
df_ciclo_n13['H01'] = (chave_2.map(dic_zp52) / 100).round(4)

df_ciclo_n13['ZP52'] = (df_ciclo_n13['H04']
                        .fillna(df_ciclo_n13['H01'])
                        .fillna(0)
                        )
                        
df_ciclo_n13['ZP52'] = df_ciclo_n13['ZP52'].round(4)

fim_zp52 = time.time()
tempo_zp52 = fim_zp52 - inicio_zp52

### ZP73

In [34]:
inicio_zp73 = time.time()

# Importação

tipos_colunas = {'CHAVE' : str, 'Cadastro': float}

colunas = ['CHAVE', 'Cadastro']

df_zp73 = pd.read_excel('../data/Arquivos/ZP73.xlsx', 
                        header=1,
                        usecols=colunas,
                        dtype=tipos_colunas,
                        engine='calamine')

df_zp73['Cadastro'] = df_zp73['Cadastro'].round(2)

df_zp73['CHAVE'] = df_zp73['CHAVE'].astype(str).str.strip()

# Chaves

dic_zp73 = df_zp73.set_index('CHAVE')['Cadastro'].to_dict()

chave_1 = df_ciclo_n13['Company Code'].astype(str) + '_' + df_ciclo_n13['COD_CLIENTE'].astype(str)
chave_2 = df_ciclo_n13['Company Code'].astype(str) + '_' + df_ciclo_n13['COD SUBREDE'].astype(str)

df_ciclo_n13['ZP73'] = ((chave_1.map(dic_zp73) / 100).round(4)
                        .fillna((chave_2.map(dic_zp73) / 100).round(4))
                        .fillna(0)
                        )
                        
df_ciclo_n13['ZP73'] = df_ciclo_n13['ZP73'].round(4)

fim_zp73 = time.time()
tempo_zp73 = fim_zp73 - inicio_zp73

### ZP70

In [35]:
inicio_zp70 = time.time()

# Importação

tipos_colunas = {'CONDICAO DE PAGAMENTO' : str, 'Desconto': float}

colunas = ['CONDICAO DE PAGAMENTO', 'Desconto']

df_zp70 = pd.read_excel('../data/Arquivos/ZP70.xlsx', 
                        header=0,
                        usecols=colunas,
                        dtype=tipos_colunas,
                        engine='calamine')

df_zp70['Desconto'] = df_zp70['Desconto'].round(4)

# Busca

dic_zp70 = df_zp70.set_index('CONDICAO DE PAGAMENTO')['Desconto'].to_dict()

chave_1 = df_ciclo_n13['COND. PAG'].astype(str)

df_ciclo_n13['ZP70'] = ((chave_1.map(dic_zp70)).round(4)
                        .fillna(0)
                        )
                        
df_ciclo_n13['ZP70'] = df_ciclo_n13['ZP70'].round(4)

fim_zp70 = time.time()
tempo_zp70 = fim_zp70 - inicio_zp70

### ZP39

In [36]:
inicio_zp39 = time.time()

# Importação

tipos_colunas = {'CHAVE' : str, 'Cadastro': float, 'P\'ANO_FIM': str}

colunas = ['CHAVE', 'Cadastro', 'P\'ANO_FIM']

df_zp39 = pd.read_excel('../data/Arquivos/ZP39.xlsx', 
                        header=1,
                        usecols=colunas,
                        dtype=tipos_colunas,
                        engine='calamine')

df_zp39['Cadastro'] = df_zp39['Cadastro'].round(4)

# Busca

dic_zp391 = df_zp39.set_index('CHAVE')['Cadastro'].to_dict()
dic_zp392 = df_zp39.set_index('CHAVE')['P\'ANO_FIM'].to_dict()

chave_1 = df_ciclo_n13['Company Code'].astype(str) + '_' + df_ciclo_n13['COD_CLIENTE'].astype(str) + '_' + df_ciclo_n13['Hierarquia'].astype(str).str[:10]

df_ciclo_n13['1. ZP39'] = ((chave_1.map(dic_zp391) / 100).round(4)
                        .fillna(0)
                        )

df_ciclo_n13['2. ZP39'] = chave_1.map(dic_zp392)

df_ciclo_n13['1. ZP39'] = df_ciclo_n13['1. ZP39'].round(4)

fim_zp39 = time.time()
tempo_zp39 = fim_zp39 - inicio_zp39

### NIV

In [37]:
inicio_niv = time.time()

# NIV/CDA

df_ciclo_n13['NIV/CDA'] = df_ciclo_n13['GSV/CDA'] * (1 + df_ciclo_n13['ZP53']) * (1 + df_ciclo_n13['ZP52']) * (1 + df_ciclo_n13['ZP73']) * (1 + df_ciclo_n13['ZP70']) * (1 + df_ciclo_n13['1. ZP39']) 

df_ciclo_n13['NIV/CDA'] = df_ciclo_n13['NIV/CDA'].round(4)

# NIV/TON

coluna_BA = df_ciclo_n13['NIV/CDA'] 
coluna_AA = df_ciclo_n13['kg/UN'] 
coluna_AO = df_ciclo_n13['Unid/CDA'] 

denominador = coluna_AA * coluna_AO

df_ciclo_n13['NIV/TON'] = np.where(
    (denominador == 0) | (denominador.isna()),
    np.nan,                                   
    (coluna_BA / denominador) * 1000           
)

df_ciclo_n13['NIV/TON'] = df_ciclo_n13['NIV/TON'].round(4)


fim_niv = time.time()
tempo_niv = fim_niv - inicio_niv

## Métricas

In [48]:
fim_total = time.time()
tempo_total = fim_total - inicio_total


print('# # # Tempos de Execução # # #\n\n'
'Ciclo: {:.2f} segundos\n'
'ZP55: {:.2f} segundos\n'
'ZP54: {:.2f} segundos\n'
'GSV: {:.2f} segundos\n'
'Projeção: {:.2f} segundos\n'
'ZP53: {:.2f} segundos\n'
'ZP52: {:.2f} segundos\n'
'ZP73: {:.2f} segundos\n'
'ZP70: {:.2f} segundos\n'
'ZP39: {:.2f} segundos\n'
'NIV: {:.2f} segundos\n'
'Total: {:.2f} segundos'
.format(tempo_ciclo, tempo_zp55, tempo_zp54, tempo_gsv, tempo_projecao, tempo_zp53, tempo_zp52, tempo_zp73, tempo_zp70, tempo_zp39, tempo_niv, tempo_total))

print('')

num_linhas = len(df_ciclo_n13)
print(f'Número de linhas do df_ciclo_n13: {num_linhas}')

numeric_sums = df_ciclo_n13.select_dtypes(include=['number']).sum(numeric_only=True)
for coluna, soma in numeric_sums.items():
    print(f'{coluna}: {soma:.4f}')
# excel = time.time()

# # df_ciclo_n13.to_excel('../data/Arquivos/Ciclo_P{:02d}_N13P_{}_processado.xlsx'.format(periodo, ano), index=False, engine='openpyxl')

# excel_fim = time.time()
# tempo_excel = excel_fim - excel

# print(f'Tempo para salvar Excel: {tempo_excel:.2f} segundos')

# # # Tempos de Execução # # #

Ciclo: 26.24 segundos
ZP55: 2.51 segundos
ZP54: 6.77 segundos
GSV: 0.01 segundos
Projeção: 0.02 segundos
ZP53: 2.82 segundos
ZP52: 0.68 segundos
ZP73: 0.34 segundos
ZP70: 0.03 segundos
ZP39: 0.50 segundos
NIV: 0.02 segundos
Total: 664.88 segundos

Número de linhas do df_ciclo_n13: 351653
P03-2026: 17935.5335
P04-2026: 14898.9761
P05-2026: 16994.5423
P06-2026: 17152.2462
P07-2026: 17810.2952
P08-2026: 18376.3242
P09-2026: 18705.5491
P10-2026: 17374.9707
P11-2026: 17610.9109
P12-2026: 17367.7446
P13-2026: 10966.4056
kg/UN: 656044.2427
Ton/CDA: 2409.5347
Unid/CDA: 8304914.0000
LSV: 84231469.7600
CLIENTE: 884.2871
CD + UF DESTINO + Importação: 260.0815
CD + UF DESTINO + NCM: 80090.0749
CD + UF DESTINO + H05: 2115.6001
ZP55: 81945.0337
1. CLIENTE: -1451.7664
1. REDE: 336.4335
1. GP UF HIER 6: -14834.5780
1. GP UF HIER 5: -189778.5734
ZP54: -203569.1855
GSV/CDA: 42021537.6971
GSV/TON: 9647773177.6222
GSV R$ P03-2026 - 2026: 186602145.3604
GSV R$ P04-2026 - 202